In [21]:
import pandas as pd
import torch
from torch_geometric.data import HeteroData
from sklearn.model_selection import train_test_split
import numpy as np

# -----------------------
# Load CSVs
# -----------------------

persons = pd.read_csv("person_vertices.csv")          # person_id + features...
conditions = pd.read_csv("condition_vertices.csv")    # condition_concept_id + features...
edges = pd.read_csv("edge_list.csv")                  # columns: person_id, condition_concept_id

# -----------------------
# Create ID maps (PyG requires consecutive IDs)
# -----------------------

person_id_map = {pid: i for i, pid in enumerate(persons['person_id'])}
condition_id_map = {cid: i for i, cid in enumerate(conditions['condition_concept_id'])}

# Map edges to contiguous IDs
edges['pid'] = edges['person_id'].map(person_id_map)
edges['cid'] = edges['condition_concept_id'].map(condition_id_map)

# make boolean columns numeric
for col in persons.columns:
    if persons[col].dtype == bool:
        persons[col] = persons[col].astype(int)

# Extract feature matrices
person_features = torch.tensor(
    persons.drop(columns=['person_id']).values, dtype=torch.float
)

# keep name column as a Python list of labels
condition_labels = conditions['concept_name'].tolist()

# numeric features only
condition_features = torch.tensor(
    conditions.drop(columns=['condition_concept_id', 'concept_name', 'occurrences']).values.astype(float),
    dtype=torch.float
)

# -----------------------
# Build HeteroData graph
# -----------------------

data = HeteroData()

data['person'].x = person_features
data['condition'].x = condition_features

edge_index = torch.tensor(edges[['pid', 'cid']].values.T, dtype=torch.long)

data['person', 'has_condition', 'condition'].edge_index = edge_index


In [22]:
# Create edge index array
num_edges = edge_index.size(1)
all_idx = np.arange(num_edges)

train_idx, test_idx = train_test_split(all_idx, test_size=0.10, random_state=42)
train_idx, val_idx  = train_test_split(train_idx, test_size=0.10, random_state=42)

train_edges = edge_index[:, train_idx]
val_edges   = edge_index[:, val_idx]
test_edges  = edge_index[:, test_idx]

# Training graph only contains train edges
data['person', 'has_condition', 'condition'].edge_index = train_edges


In [23]:
def negative_sampling(num_neg, num_persons, num_conditions, positive_set):
    """Return random negative edges (pid, cid) not in positive_set."""
    neg_p = np.random.randint(0, num_persons, size=num_neg)
    neg_c = np.random.randint(0, num_conditions, size=num_neg)
    neg = list(zip(neg_p, neg_c))

    # Filter out positives
    neg = [e for e in neg if e not in positive_set]
    neg = neg[:num_neg]   # trim if needed

    return torch.tensor(np.array(neg).T, dtype=torch.long)


In [24]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, to_hetero

class BaseGNN(nn.Module):
    def __init__(self, conv):
        super().__init__()
        self.conv1 = conv
        self.conv2 = conv

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x


def get_model(model_type, metadata, hidden=64):
    if model_type == "gcn":
        conv = GCNConv
    elif model_type == "sage":
        conv = SAGEConv
    elif model_type == "gat":
        conv = lambda in_c, out_c: GATConv(in_c, out_c, heads=2)
    else:
        raise ValueError("Unknown model type")

    base = BaseGNN(conv(-1, hidden))  # -1 = infer input_feats
    model = to_hetero(base, metadata, aggr='sum')

    return model


In [25]:
def edge_score(node_emb, edge_index):
    src, dst = edge_index
    return (node_emb[src] * node_emb[dst]).sum(dim=-1)


In [26]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score

def train(model, data, train_edges, num_persons, num_conditions):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    pos_edges_set = set(zip(train_edges[0].tolist(), train_edges[1].tolist()))

    for epoch in range(50):
        optimizer.zero_grad()

        # forward pass
        out = model(data.x_dict, data.edge_index_dict)

        pos_score = edge_score(out['person'], out['condition'], train_edges)

        neg_edges = negative_sampling(
            num_neg=len(train_edges[0]),
            num_persons=num_persons,
            num_conditions=num_conditions,
            positive_set=pos_edges_set
        )
        neg_score = edge_score(out['person'], out['condition'], neg_edges)

        # labels
        y = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
        pred = torch.cat([pos_score, neg_score])

        loss = nn.BCEWithLogitsLoss()(pred, y)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


In [27]:
def evaluate(model, data, pos_edges, num_persons, num_conditions):
    model.eval()
    out = model(data.x_dict, data.edge_index_dict)

    pos_score = torch.sigmoid(edge_score(out['person'], out['condition'], pos_edges))

    # sample equal number of negatives
    neg_edges = negative_sampling(
        len(pos_edges[0]),
        num_persons,
        num_conditions,
        positive_set=set(zip(pos_edges[0].tolist(), pos_edges[1].tolist()))
    )
    neg_score = torch.sigmoid(edge_score(out['person'], out['condition'], neg_edges))

    y_true = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)]).cpu()
    y_pred = torch.cat([pos_score, neg_score]).detach().cpu()

    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    return auc, ap


In [ ]:
num_persons = len(persons)
num_conditions = len(conditions)

metadata = data.metadata()

for model_type in ["gcn", "sage", "gat"]:
    print(f"\n=== Training {model_type.upper()} ===")

    model = get_model(model_type, metadata)
    train(model, data, train_edges, num_persons, num_conditions)

    auc, ap = evaluate(model, data, val_edges, num_persons, num_conditions)
    print(f"{model_type.upper()} | Val AUC: {auc:.4f} | AP: {ap:.4f}")
    
    auc, ap = evaluate(model, data, test_edges, num_persons, num_conditions)
    print(f"Test AUC: {auc:.4f} | AP: {ap:.4f}")




=== Training GCN ===


c:\Users\patri\Desktop\code\NetworkAnalysis\.venv\Lib\site-packages\torch_geometric\nn\to_hetero_transformer.py:151: UserWarning: There exist node types ({'person'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  self.validate()


ValueError: 'add_self_loops' attribute set to 'True' on module 'GCNConv(-1, 64)' for use with edge type(s) '[('person', 'has_condition', 'condition')]'. This will lead to incorrect message passing results.